In [ ]:
# Name: Lame Wadipeba
# Email: wadipeba@usc.edu

In [ ]:
# Install Java
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
# Install PySpark (if not already installed)
!pip install pyspark -qq > /dev/null

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as fc

# Start a Spark session
spark = SparkSession.builder.appName("demo").getOrCreate()


In [ ]:
#Reading in json files into databases
yelp_business = spark.read.json("yelp_business.json")
yelp_users = spark.read.json("yelp_users_cleaned.json")

yelp_business.show(5)
yelp_users.show(5)


+--------------------+--------------------+--------------------+--------------------+---------+--------------------+-------+-------------+--------------+-------------------+--------------+-----------+------------+-----+-----+--------+
|             address|          attributes|         business_id|          categories|     city|               hours|is_open|     latitude|     longitude|               name|  neighborhood|postal_code|review_count|stars|state|    type|
+--------------------+--------------------+--------------------+--------------------+---------+--------------------+-------+-------------+--------------+-------------------+--------------+-----------+------------+-----+-----+--------+
|227 E Baseline Rd...|[BikeParking: Tru...|0DI8Dt2PJp07XkVvI...|[Tobacco Shops, N...|    Tempe|[Monday 11:0-21:0...|      0|   33.3782141|   -111.936102|  Innovative Vapors|              |      85283|          17|  4.5|   AZ|business|
|495 S Grand Centr...|[BusinessAcceptsB...|LTlCaCGZE14GuaUXU

In [ ]:
#5a Find users who have written more than 200 reviews but have an
#average rating below 3.5. Show their user_id, name, review_count
#and average_star.
yelp_users.filter("review_count > 200 AND average_stars < 3.5").select("user_id", "name", "review_count", "average_stars").show()

+--------------------+------------------+------------+-------------+
|             user_id|              name|review_count|average_stars|
+--------------------+------------------+------------+-------------+
|ZMocFA2lNMJeiKjgu...|              Jack|        1005|         3.48|
|lBIfa3INjAj_z2_am...|             Diane|         978|         3.08|
|iC5JO52dGKJNxfzp3...|            Barney|         983|         3.26|
|Yd0Do7Ki_GQlUdLKu...|             James|        1015|         3.49|
|rLl5jjjUK1crVfzbX...|               Jay|         278|         3.31|
|e3fX7_qkoSm-6-yTd...|           Riza H.|        1143|         3.32|
|AZUF5EVwh-W08nrFT...|              John|         218|         3.38|
|D3kM83KJ-0TceDEOy...|           Patrick|         460|          3.4|
|7yU76bRb0HuPVWeTT...|            Rodene|         216|         2.43|
|5gz8szYeYntJMCBtR...|            Angela|         504|         3.46|
|3vGiPNH4yCAj88-kf...|              Noha|         212|         3.44|
|rUzp2KsuzpjXy_D-L...|Allison Wond

In [ ]:
#5b Find the user with an average star rating greater than 4.5 who gave
#the most funny compliments. Show his/her user_id, name,
#compliment_funny, and average_stars.
yelp_users.filter("average_stars > 4.5").\
orderBy("compliment_funny", ascending=False).\
select("user_id", "name", "compliment_funny", "average_stars").\
limit(1).show()

+--------------------+----+----------------+-------------+
|             user_id|name|compliment_funny|average_stars|
+--------------------+----+----------------+-------------+
|IpReaQYuXO_nB07xn...|Mary|             590|         4.53|
+--------------------+----+----------------+-------------+



In [ ]:
#5c Find how many users joined Yelp in each year. Hint: Use
#fc.year("yelping_since") to extract the year from the date.
yelp_users.groupBy(fc.year("yelping_since").alias("year_joined")).\
agg(fc.count("*").alias("num_of_users")).\
orderBy("year_joined").\
select("year_joined","num_of_users").show()

+-----------+------------+
|year_joined|num_of_users|
+-----------+------------+
|       2004|           2|
|       2005|          32|
|       2006|         137|
|       2007|         388|
|       2008|         691|
|       2009|         962|
|       2010|        1412|
|       2011|        1628|
|       2012|        1476|
|       2013|        1292|
|       2014|        1055|
|       2015|         716|
|       2016|         209|
+-----------+------------+



In [ ]:
#5d Find the average number of reviews grouped by whether users have
#fans or not. Hint: You can refer to Exercise 2 in the lecture slides.
#Use .withColumn() and fc.lit() to label groups as 'Has Fans' and 'No
#Fans'. Use union() to combine the two results before displaying.

# has fans
has_fans = yelp_users.filter("fans > 0").\
agg(fc.avg("review_count").alias("avg_reviews")).\
withColumn("tier", fc.lit("Has_fans"))

# no fans
no_fans = yelp_users.filter("fans == 0").\
agg(fc.avg("review_count").alias("avg_reviews")).\
withColumn("tier", fc.lit("No_fans"))

#union
has_fans.union(no_fans).select("tier", "avg_reviews").show()

+--------+-----------------+
|    tier|      avg_reviews|
+--------+-----------------+
|Has_fans|175.3956315681022|
| No_fans|14.08082378084321|
+--------+-----------------+



In [ ]:
#6a
data = [("Akk3JwQ4Rv9v0cm9u8jtMw", "Mi5uhdFB9OJteXPd0_IKfw", 4.0, "Nice Italian food"),
("Q1drYdla0MTluxFqbZsmmw", "YPavuOh2XsnRbLfl0DH2lQ", 5.0, "Best chicken ever"),
("eCPeMIEAjOpzq5ArgS_tBg", "93otbGHE0s0m-lU1osvg9w", 4.5, "Cozy place"),
("ZMocFA2lNMJeiKjguUKu7g", "L_thK7r3K_h5M4tV7amEKQ", 3.5, "Good, but slow service"),
("dJtFy2jc1R5cSBab4BLORQ", "V94gk_Vx0Ot623UGHG6PrA", 4.8, "Love the drinks"),
("DIIPKuuw4vDBeo_uXj-uSA", "4tzmCT04wTAbbiZFKo2cPQ", 2.5, "Poor experience"),
("56PMGk12KYLJWPt7ejg-XA", "mvsnt4q3A5TzcbPFolpsZQ", 4.2, "Good family place"),
("lXYuAREWWD6pwu9Y0R4anQ", "S6HTXcS97j173_btjN8w7Q", 4.7, "Tasty BBQ ribs"),
("abcdefghijkl1234567890", "Mi5uhdFB9OJteXPd0_IKfw", 3.0, "Service was okay"),
("Akk3JwQ4Rv9v0cm9u8jtMw", "FAKE_BIZ_1", 4.2, "No comment")]
schema = ["user_id", "business_id", "rating", "comment"]

#Create a new DataFrame named user_review that contains the
#following information and print the DataFrame.
user_review = spark.createDataFrame(data,schema)
user_review.show()


+--------------------+--------------------+------+--------------------+
|             user_id|         business_id|rating|             comment|
+--------------------+--------------------+------+--------------------+
|Akk3JwQ4Rv9v0cm9u...|Mi5uhdFB9OJteXPd0...|   4.0|   Nice Italian food|
|Q1drYdla0MTluxFqb...|YPavuOh2XsnRbLfl0...|   5.0|   Best chicken ever|
|eCPeMIEAjOpzq5Arg...|93otbGHE0s0m-lU1o...|   4.5|          Cozy place|
|ZMocFA2lNMJeiKjgu...|L_thK7r3K_h5M4tV7...|   3.5|Good, but slow se...|
|dJtFy2jc1R5cSBab4...|V94gk_Vx0Ot623UGH...|   4.8|     Love the drinks|
|DIIPKuuw4vDBeo_uX...|4tzmCT04wTAbbiZFK...|   2.5|     Poor experience|
|56PMGk12KYLJWPt7e...|mvsnt4q3A5TzcbPFo...|   4.2|   Good family place|
|lXYuAREWWD6pwu9Y0...|S6HTXcS97j173_btj...|   4.7|      Tasty BBQ ribs|
|abcdefghijkl12345...|Mi5uhdFB9OJteXPd0...|   3.0|    Service was okay|
|Akk3JwQ4Rv9v0cm9u...|          FAKE_BIZ_1|   4.2|          No comment|
+--------------------+--------------------+------+--------------

In [ ]:
#6b Use an inner join to join user_review with yelp_users on the column
#user_id. Display each reviewer’s user_id, name, rating, and comment.
user_review.join(yelp_users, on="user_id", how="inner").\
select("user_id", "name", "rating", "comment").show()

+--------------------+-------+------+--------------------+
|             user_id|   name|rating|             comment|
+--------------------+-------+------+--------------------+
|Akk3JwQ4Rv9v0cm9u...|melissa|   4.0|   Nice Italian food|
|Q1drYdla0MTluxFqb...| Amelia|   5.0|   Best chicken ever|
|eCPeMIEAjOpzq5Arg...| Janine|   4.5|          Cozy place|
|ZMocFA2lNMJeiKjgu...|   Jack|   3.5|Good, but slow se...|
|dJtFy2jc1R5cSBab4...|  Bryan|   4.8|     Love the drinks|
|DIIPKuuw4vDBeo_uX...|  Milke|   2.5|     Poor experience|
|56PMGk12KYLJWPt7e...|Crystal|   4.2|   Good family place|
|lXYuAREWWD6pwu9Y0...| Austin|   4.7|      Tasty BBQ ribs|
|Akk3JwQ4Rv9v0cm9u...|melissa|   4.2|          No comment|
+--------------------+-------+------+--------------------+



In [ ]:
#6c Use a left join to identify which reviews in user_review were written
#by users not found in the yelp_users dataset. Show user_id, rating,
#and comment.
user_review.join(yelp_users, on = "user_id", how="left").\
select("user_id","comment","rating").show()

+--------------------+--------------------+------+
|             user_id|             comment|rating|
+--------------------+--------------------+------+
|Akk3JwQ4Rv9v0cm9u...|   Nice Italian food|   4.0|
|Q1drYdla0MTluxFqb...|   Best chicken ever|   5.0|
|eCPeMIEAjOpzq5Arg...|          Cozy place|   4.5|
|ZMocFA2lNMJeiKjgu...|Good, but slow se...|   3.5|
|dJtFy2jc1R5cSBab4...|     Love the drinks|   4.8|
|DIIPKuuw4vDBeo_uX...|     Poor experience|   2.5|
|56PMGk12KYLJWPt7e...|   Good family place|   4.2|
|lXYuAREWWD6pwu9Y0...|      Tasty BBQ ribs|   4.7|
|abcdefghijkl12345...|    Service was okay|   3.0|
|Akk3JwQ4Rv9v0cm9u...|          No comment|   4.2|
+--------------------+--------------------+------+



In [ ]:
#6d Display each reviewer’s name, the business name, the city, and the
#rating from the user_review DataFrame. Only include valid reviews
#— those where both the user_id and business_id exist in the joined
#tables. Rename the output columns as user_name, business_name,
#city, and rating.
user_review.alias("ur").join(yelp_users.alias("yu"), on = "user_id", how = "inner").\
join(yelp_business.alias("yb"), on = "business_id", how = "inner").\
select(fc.col("yu.name").alias("user_name"), fc.col("yb.name").alias("business_name"),
       fc.col("yb.city").alias("city"),fc.col("ur.rating").alias("rating")).show()

SyntaxError: invalid character '—' (U+2014) (ipython-input-2289297937.py, line 3)

In [ ]:
#7a Create RDDs manually for both Yelp business and Yelp user
#datasets
business_rdd = yelp_business.rdd
users_rdd = yelp_users.rdd

In [ ]:
#7b Using business_rdd, show the number of businesses per city,
# showing the top 10 cities by count.
city_counts = (business_rdd.map(lambda x: ( x.city, 1)).reduceByKey(lambda x, y: x + y)
    .sortBy(lambda x: x[1], ascending=False).take(10))
city_counts

[('Las Vegas', 22892),
 ('Toronto', 14540),
 ('Phoenix', 14468),
 ('Scottsdale', 6917),
 ('Charlotte', 6912),
 ('Pittsburgh', 5275),
 ('Montréal', 4785),
 ('Mesa', 4714),
 ('Henderson', 3788),
 ('Tempe', 3703)]

In [ ]:
#7c Using user_rdd, find users who have more than 250 reviews and
#average stars above 4.5. This question is similar to 5a, but must be
#done in RDD format. Make sure to output user_id, name,
#review_count and average_stars
good = (users_rdd.filter(lambda x: x.review_count > 250 and x.average_stars > 4.5)
    .map(lambda x: (x.user_id, x.name, x.review_count, x.average_stars)))
good.collect()

[('-UtSFRnsdUgC_R1OzGZVhA', 'Kamilah', 394, 4.64),
 ('TlMK-Ad_NLIHCX1XCwrqSw', 'Dominique', 301, 4.71),
 ('7sNE58P4AvsX6QHE8ypCiA', 'Sheila', 440, 4.53),
 ('VTMJ7f2EClw_WOM_P-sDgw', 'Miguel', 269, 4.52),
 ('V7B4KTBitlBs9YH_B00-9Q', 'Matthew', 304, 4.58),
 ('IpReaQYuXO_nB07xnkHuNw', 'Mary', 655, 4.53)]